# Solutions — Notebook 01: Introduction and Setup

**BINFX410 — Chapter 10**

> **Prerequisite:** Run `01_introduction_and_setup.ipynb` first to generate the `../raw_data/` CSV files. The setup cell below reads those files directly.

In [ ]:
import pandas as pd

df_samples  = pd.read_csv('../raw_data/samples.csv')
df_genes    = pd.read_csv('../raw_data/genes.csv')
df_calls    = pd.read_csv('../raw_data/variant_calls.csv')
df_variants = pd.read_csv('../raw_data/variants.csv')

print('Data loaded.')
print(f'  df_samples  : {len(df_samples):,} rows')
print(f'  df_genes    : {len(df_genes):,} rows')
print(f'  df_calls    : {len(df_calls):,} rows')
print(f'  df_variants : {len(df_variants):,} rows')

## Exercise 1.1 — Explore the Raw Data

Using the DataFrames already in memory (`df_samples`, `df_genes`, `df_calls`, `df_variants`), answer:

a) What is the average variant sequencing depth (`depth`)?

b) How many unique samples have at least one variant call with `filter_status = 'PASS'`?

c) What fraction of variant calls have a non-PASS filter status (i.e., `filter_status != 'PASS'`)?  

d) Which gene biotype has the highest mean `quality_score` across all variants? (Hint: join `df_variants` to `df_genes` on `gene_id`.)

In [ ]:
# a) Average variant sequencing depth
avg_depth = df_variants['depth'].mean()
print(f'a) Average variant depth: {avg_depth:.1f}x')

# b) Unique samples with at least one PASS variant call
pass_call_ids    = df_calls[df_calls['filter_status'] == 'PASS']['call_id']
pass_variants    = df_variants[df_variants['call_id'].isin(pass_call_ids)]
pass_sample_ids  = df_calls[df_calls['call_id'].isin(pass_variants['call_id'])]['sample_id']
unique_samples_pass = pass_sample_ids.nunique()
print(f'b) Unique samples with \u22651 PASS variant call: {unique_samples_pass}')

# c) Fraction of non-PASS variant calls
non_pass_frac = (df_calls['filter_status'] != 'PASS').mean()
print(f'c) Non-PASS fraction of variant calls: {non_pass_frac:.3f} ({non_pass_frac*100:.1f}%)')

# d) Biotype with highest mean quality score
# Join variants to genes to get biotype
variants_with_biotype = df_variants.merge(df_genes[['gene_id', 'biotype']], on='gene_id', how='left')
top_biotype = variants_with_biotype.groupby('biotype')['quality_score'].mean().idxmax()
top_quality  = variants_with_biotype.groupby('biotype')['quality_score'].mean().max()
print(f'd) Biotype with highest mean quality: {top_biotype} (mean quality {top_quality:.2f})')

## Exercise 1.2 — Star Schema Design

If you were designing a data warehouse star schema for this genomics dataset, what would your fact and dimension tables look like?

---

**Fact table:** `fact_variants` (grain = one row per variant)

**Measures:** `af_tumor`, `depth`, `quality_score`, `signal_strength` (depth × af_tumor)

**Dimension tables:**

```
dim_date         dim_samples          dim_genes
────────         ───────────          ─────────
date_key (PK)    sample_key (PK)      gene_key (PK)
full_date        sample_id            gene_id
year             patient_id           gene_name
month            tissue_type          chromosome
quarter          sequencing_platform  biotype
...              library_prep         is_cancer_gene
                 project_id           gene_length
                 collection_date
                 mean_coverage

fact_variant_calls          fact_variants (fine-grain)
──────────────────          ──────────────────────────
call_key (PK)               variant_key (PK)
call_id                     variant_id
sample_key (FK)             call_key (FK)
date_key (FK)               sample_key (FK)
caller                      gene_key (FK)
filter_status               date_key (FK)
variant_count               variant_type
mean_quality                consequence
                            af_tumor      ← measure
                            depth         ← measure
                            quality_score ← measure
                            signal_strength ← measure
```

**Why this grain?** The variant is the finest-grain event in the pipeline. Analysts can always roll up to call, sample, gene, or time — but can't drill down from a coarser grain.

## Exercise 1.3 — Architecture Selection

For each scenario below, which architecture (warehouse, lake, or lakehouse) would you recommend, and why?

---

**a) Hospital: monthly reports on somatic mutation burden per patient (structured EMR + variant data)**

**Recommendation: Data Warehouse**

The workload is structured, schema-stable reporting with known query patterns (monthly aggregates, patient cohorts). Schema-on-write catches data quality problems before they reach reports — critical in a clinical setting. DuckDB can host the star schema locally with zero infrastructure. A data lake would add schema-on-read overhead without benefit for this well-defined query pattern.

---

**b) Sequencing lab: raw FASTQ from 200 runs/week, custom Python exploration**

**Recommendation: Data Lake (medallion architecture)**

FASTQ files are unstructured binary data that don't fit a relational schema. Researchers need ad-hoc access patterns that can't be predicted at schema design time. The Bronze zone preserves every raw file immutably; Silver can hold processed VCF/variant tables; Gold holds curated summary statistics. PyArrow + Parquet gives researchers fast columnar access for custom scripts without requiring a database server.

---

**c) Biotech: SQL dashboards for management + ML pathogenicity scoring models on the same dataset**

**Recommendation: Lakehouse**

This workload explicitly requires two different access patterns on the same data: SQL analytics (OLAP) and ML model training (Python DataFrames, feature extraction). A plain warehouse can't serve ML workloads natively. A plain lake lacks the ACID guarantees needed for reliable dashboards. The lakehouse (Delta Lake) gives both: `delta_scan()` for SQL via DuckDB, and `.to_pandas()` / `.to_pyarrow()` for ML pipelines — all reading the same Parquet files with ACID consistency.